# 02 — Baseline detector training (v1)

Giai đoạn 2 (`docs/PLAN.md`) — spec: `docs/specs/g2-baseline-training.md`.

Mục tiêu: train baseline YOLOv8 (`yolov8n.pt`) trên dataset v1 (5 lớp, 1013
ảnh — số liệu thật từ Giai đoạn 1), ghi lại mAP@0.5 / mAP@0.5:0.95 làm mốc
so sánh cho ablation (Giai đoạn 3) và retrain sau cải tiến (Giai đoạn 5).

**Chạy trên Colab** (cần GPU — Runtime → Change runtime type → T4 GPU).

## Setup — mount Drive + cd vào repo

Giống hệt cell setup ở `01_data_exploration.ipynb` — bắt buộc để
`weights/best.pt` train ra tự động nằm trong Drive (không mất khi hết
session), và để `data/raw/yoga_v1/` (đã tải ở Giai đoạn 1) tìm thấy được.

In [ ]:
import os

REPO_DIR_NAME = "computer-vision-project"  # đổi nếu bạn git clone ra tên thư mục khác

try:
    from google.colab import drive

    drive.mount("/content/drive")
    drive_path = f"/content/drive/MyDrive/{REPO_DIR_NAME}"
    if not os.path.isdir(drive_path):
        raise FileNotFoundError(
            f"{drive_path} không tồn tại — kiểm tra lại bạn đã `git clone` repo vào "
            "đúng chỗ trong Drive chưa (T0.4), hoặc sửa REPO_DIR_NAME ở trên cho khớp."
        )
    os.chdir(drive_path)
except ImportError:
    pass  # không chạy trên Colab (vd Jupyter local) — giả định cwd đã là repo root

print("cwd:", os.getcwd())
assert os.path.isdir("scripts") and os.path.isdir("data"), (
    "Chưa đứng ở repo root — không thấy scripts/ và data/ ở cwd hiện tại."
)
assert os.path.isfile("data/raw/yoga_v1/data.yaml"), (
    "data/raw/yoga_v1/data.yaml chưa có — chạy notebooks/01_data_exploration.ipynb "
    "(Giai đoạn 1) trước để tải dataset."
)

## Đồng bộ code mới nhất

Thư mục Drive được clone 1 lần ở T0.4 — nếu code trên GitHub đã cập nhật
sau đó (vd `src/models/train.py` mới thêm), cell này `git pull` để lấy về.
An toàn chạy lại nhiều lần.

In [ ]:
import os

assert os.path.isdir(".git"), (
    "cwd hiện tại không phải repo root (không thấy .git) — runtime Colab có "
    "thể vừa bị reset. Chạy lại cell 'Setup — mount Drive + cd vào repo' ở "
    "trên (mount + cd) trước, rồi chạy lại cell này."
)
!git pull

## Cài dependencies

Mỗi phiên Colab mới đều mất hết package đã cài — cell này idempotent, chạy
lại nhiều lần không sao.

In [ ]:
!pip install -q -r requirements.txt
import torch
import ultralytics

print("ultralytics", ultralytics.__version__, "| torch", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print(
        "CẢNH BÁO: không thấy GPU — vào Runtime > Change runtime type > "
        "chọn T4 GPU, rồi chạy lại từ đầu notebook. Train trên CPU sẽ rất chậm."
    )

## T2.1–T2.2 — Train baseline

`train()` (`src/models/train.py`) là wrapper mỏng quanh
`ultralytics.YOLO(...).train(...)`, cố định seed + augmentation đã chốt ở
T1.5. Baseline: `yolov8n.pt`, `seed=42`, `epochs=50`, `imgsz=640` (lý do chốt
epochs=50 xem `docs/specs/g2-baseline-training.md`).

In [ ]:
from src.models.train import train

results = train(
    data="data/raw/yoga_v1/data.yaml",
    model="yolov8n.pt",
    seed=42,
    epochs=50,
    name="yolov8n_v1_baseline",
    exist_ok=True,  # rerun ghi đè cùng thư mục thay vì tự tăng hậu tố (_2, _3, ...)
)
print("save_dir:", results.save_dir)

## T2.3 — Xác nhận output

In [ ]:
from pathlib import Path

save_dir = Path(results.save_dir)
expected = ["results.csv", "results.png", "weights/best.pt", "weights/last.pt"]
for rel in expected:
    p = save_dir / rel
    print(f"{'OK ' if p.exists() else 'MISSING'}  {p}")

## T2.4 — Weights nằm trong Drive

Vì cell Setup ở đầu notebook đã `cd` vào repo trong Drive, `results.save_dir`
phải tự động nằm dưới `/content/drive/...` — không cần copy tay.

In [ ]:
resolved = save_dir.resolve()
print("save_dir (resolved):", resolved)
if "drive" in str(resolved).lower():
    print("OK — nằm trong Google Drive, không mất khi hết session.")
else:
    print(
        "CẢNH BÁO: save_dir không có vẻ nằm trong Drive — kiểm tra lại cell "
        "Setup ở đầu notebook đã cd đúng chỗ chưa trước khi train."
    )

## T2.5 — mAP baseline

`train()` đã tự chạy 1 lượt validate trên `best.pt` ở cuối training —
`results` (cell T2.1–T2.2) chính là kết quả đó, không cần load lại
checkpoint và validate thêm lần nữa (tốn thời gian session Colab free-tier
vô ích).

In [ ]:
print("mAP@0.5:", results.box.map50)
print("mAP@0.5:0.95:", results.box.map)

**mAP baseline (chạy thật trên Colab, T4 GPU, 50 epochs, `yolov8n.pt`):**

- mAP@0.5: 0.9923
- mAP@0.5:0.95: 0.8435

Rất cao cho baseline — dataset v1 tương đối "dễ" (5 lớp, tư thế khác biệt rõ
rệt về hình dạng, học viên thường chiếm phần lớn khung hình). Đây là mốc so
sánh cho ablation (Giai đoạn 3) và retrain sau cải tiến (Giai đoạn 5).

## T3.1 — Ablation: augmentation ON vs OFF

Giai đoạn 3 (`docs/PLAN.md`) — spec: `docs/specs/g3-ablation.md`. Biến duy
nhất thay đổi giữa 2 lần train: **augmentation**, giữ nguyên `model`,
`seed=42`, `epochs=50`, `imgsz=640`.

- **ON** = chính baseline ở trên (mặc định `train()`: `flipud=0.0,
  fliplr=0.5, degrees=10.0`, mọi augmentation khác — mosaic, hsv,
  translate, scale, shear, perspective, mixup, copy_paste, erasing — dùng
  mặc định Ultralytics).
- **OFF** = tắt **toàn bộ** augmentation (không chỉ 3 tham số T1.5) để có
  tương phản đủ rõ.

## T3.2 — Train variant OFF

Không train lại baseline (đã có kết quả thật ở T2.5, và `results.csv` của
nó vẫn nằm trên Drive) — chỉ train thêm variant OFF, `name` khác để không
ghi đè.

In [ ]:
results_noaug = train(
    data="data/raw/yoga_v1/data.yaml",
    model="yolov8n.pt",
    seed=42,
    epochs=50,
    name="yolov8n_v1_ablation_noaug",
    exist_ok=True,  # rerun ghi đè cùng thư mục thay vì tự tăng hậu tố (_2, _3, ...)
    # Tắt toàn bộ augmentation — xem định nghĩa OFF ở T3.1.
    flipud=0.0,
    fliplr=0.0,
    degrees=0.0,
    translate=0.0,
    scale=0.0,
    shear=0.0,
    perspective=0.0,
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,
    mosaic=0.0,
    mixup=0.0,
    copy_paste=0.0,
    erasing=0.0,
    auto_augment=None,
)
print("save_dir:", results_noaug.save_dir)

## T3.3 — Bảng so sánh baseline vs variant

Đọc trực tiếp từ `results.csv` trên đĩa của cả 2 run (không dùng biến RAM
`results`/`results_noaug` — nếu Colab session đã restart từ lúc train
baseline ở Giai đoạn 2, biến đó không còn, nhưng file trên Drive vẫn còn).

In [ ]:
from pathlib import Path

import pandas as pd

RUNS_DIR = Path("runs/detect")


def last_epoch_stats(run_name: str) -> dict:
    df = pd.read_csv(RUNS_DIR / run_name / "results.csv")
    df.columns = df.columns.str.strip()
    last = df.iloc[-1]
    return {
        "run": run_name,
        "mAP@0.5": last["metrics/mAP50(B)"],
        "mAP@0.5:0.95": last["metrics/mAP50-95(B)"],
        "train_time_s": last["time"],
    }


comparison = pd.DataFrame(
    [
        last_epoch_stats("yolov8n_v1_baseline"),
        last_epoch_stats("yolov8n_v1_ablation_noaug"),
    ]
).set_index("run")
comparison

## T3.4 — Config chính thức

**Bảng so sánh (số thật từ `comparison` ở trên):**

| Run | mAP@0.5 | mAP@0.5:0.95 | Thời gian train (s) |
|---|---|---|---|
| ON (baseline) | 0.9922 | 0.8352 | 913.2 |
| OFF (no aug) | 0.9754 | 0.8558 | 672.1 |

**Kết luận:** Kết quả trái chiều — ON thắng mAP@0.5 (+1.7pp), OFF thắng
mAP@0.5:0.95 (+2.1pp) và train nhanh hơn ~26%. Không bên nào vượt trội rõ
rệt, nên **chọn augmentation ON làm config chính thức** (giữ nguyên
`flipud=0.0, fliplr=0.5, degrees=10.0` + augmentation mặc định
Ultralytics) — lý do: mục tiêu triển khai thực tế là chạy trên ảnh/video
thật ngoài dataset (camera điện thoại, góc chụp/ánh sáng đa dạng hơn tập
train), augmentation giúp tổng quát hoá tốt hơn dù không thắng tuyệt đối
trên test set cùng phân bố. Config này dùng cho Giai đoạn 5 (retrain sau
cải tiến).

*Lưu ý: mAP baseline ở đây (0.9922/0.8352) lệch nhẹ so với lần train đầu
tiên ở T2.5 (0.9923/0.8435) dù cùng `seed=42` — do YOLO/cuDNN không hoàn
toàn deterministic giữa các lần train trên GPU (không set
`deterministic=True`), chênh lệch ở mức nhiễu bình thường, không phải
lỗi.*